# 4a. Adaptation Analysis

**Purpose**: Characterise how animals adapt when the stimulus distribution
shifts. This is the central scientific question — PPC is hypothesised to
be necessary when the internal statistical model is inadequate.

**Protocol**:
1. Detect distribution shifts in each animal's timeline
2. Split sessions into baseline (expert Uniform) and post-shift phases
3. Compute adaptation trajectories (session-by-session stats)
4. Fit recovery curves (exponential convergence)
5. Statistical comparison: baseline vs early-post vs late-post
6. Quantify shift magnitude and convergence speed
7. Aggregate across animals

**Paradigm**: Uniform → Hard-A → Hard-B → Hard-A (if time permits)

Each transition provides a different test:
- Uniform → Hard-A: first novel shift
- Hard-A → Hard-B: shift with meta-experience
- Hard-B → Hard-A: return to familiar distribution

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
from pathlib import Path

from behav_utils.data.loading import load_experiment
from behav_utils.data.selection import select_sessions
from behav_utils.plotting.styles import COLOURS, apply_style
from behav_utils.plotting.update_matrix import plot_update_matrix
from behav_utils.analysis.update_matrix import compute_update_matrix, matrix_error

from analysis.adaptation import (
    detect_all_manipulations,
    detect_first_manipulation,
    adaptation_trajectory,
    fit_recovery_curve,
    compare_phases,
    compute_shift_magnitude,
    compute_convergence_metrics,
    classify_sessions,
    aggregate_trajectories,
)

apply_style()
print('Imports OK')

## 1. Configuration

In [ ]:
CONFIG_PATH = Path('../config.yaml')
STAGE = 'Full_Task_Cont'

# Baseline selection criteria (expert Uniform)
BASELINE_MIN_ACCURACY = 0.70
BASELINE_LAST_FRACTION = 0.50

# Stats to track across the shift
TRACK_STATS = ['accuracy', 'pse', 'slope', 'recency', 'side_bias', 'choice_entropy']

# Phase comparison windows
EARLY_POST_SESSIONS = 5   # first N post-shift sessions
LATE_POST_SESSIONS = 5    # last N post-shift sessions

## 2. Load Data & Detect Shifts

In [ ]:
experiment = load_experiment(CONFIG_PATH)
print(f"Loaded {len(experiment.animal_ids)} animals")

# Detect all manipulations across all animals
shift_info = {}  # animal_id -> list of manipulation dicts

for aid in experiment.animal_ids:
    animal = experiment.get_animal(aid)
    manips = detect_all_manipulations(animal, stage=STAGE)
    if manips:
        shift_info[aid] = manips
        for m in manips:
            print(f"  {aid}: {m['type']} at session {m['global_session_idx']} "
                  f"({m['details']})")
    else:
        print(f"  {aid}: no manipulation detected")

animals_with_shifts = list(shift_info.keys())
print(f"\n{len(animals_with_shifts)} animals with distribution shifts")

## 3. Per-Animal Adaptation Analysis

For each animal with a shift:
- Baseline = expert Uniform sessions before the shift
- Post = all sessions after the shift
- Compute trajectory, recovery curve, phase comparison

In [ ]:
per_animal_results = {}  # animal_id -> dict of results

for aid in animals_with_shifts:
    animal = experiment.get_animal(aid)
    manips = shift_info[aid]
    first_shift = manips[0]
    shift_idx = first_shift['session_idx']  # index within stage-filtered sessions

    # Get all sessions for this stage
    all_sessions = animal.get_sessions(stage=STAGE)

    # Baseline: expert Uniform sessions before the shift
    pre_sessions = all_sessions[:shift_idx]
    baseline = select_sessions(
        animal,
        stage=STAGE,
        distribution='Uniform',
        min_accuracy=BASELINE_MIN_ACCURACY,
        last_fraction=BASELINE_LAST_FRACTION,
    )
    # Only keep baseline sessions that are actually before the shift
    baseline = [s for s in baseline if s.session_idx < all_sessions[shift_idx].session_idx]

    # Post: all sessions after the shift
    post = all_sessions[shift_idx:]

    if len(baseline) < 3 or len(post) < 3:
        print(f"  {aid}: insufficient sessions "
              f"(baseline={len(baseline)}, post={len(post)}), skipping")
        continue

    print(f"\n{'='*60}")
    print(f"  {aid}: {first_shift['details']}")
    print(f"  Baseline: {len(baseline)} sessions, Post: {len(post)} sessions")

    # ── Trajectory ──────────────────────────────────────────────────────────
    traj = adaptation_trajectory(baseline, post, stats=TRACK_STATS)

    # ── Recovery curves ─────────────────────────────────────────────────────
    recovery = {}
    for stat in TRACK_STATS:
        recovery[stat] = fit_recovery_curve(baseline, post, stat=stat)

    # ── Convergence metrics ─────────────────────────────────────────────────
    convergence = compute_convergence_metrics(baseline, post, stats=TRACK_STATS)

    # ── Phase comparison ────────────────────────────────────────────────────
    early_post = post[:EARLY_POST_SESSIONS]
    late_post = post[-LATE_POST_SESSIONS:] if len(post) >= LATE_POST_SESSIONS else post

    comp_early = compare_phases(
        baseline, early_post, stats=TRACK_STATS, test='mannwhitneyu',
    )
    comp_late = compare_phases(
        baseline, late_post, stats=TRACK_STATS, test='mannwhitneyu',
    )

    # ── Shift magnitude ─────────────────────────────────────────────────────
    acc_drop = compute_shift_magnitude(baseline, post, metric='accuracy_drop')
    pse_shift = compute_shift_magnitude(baseline, post, metric='pse_shift')

    # ── State classification ────────────────────────────────────────────────
    labels = classify_sessions(baseline, post, stats=TRACK_STATS)

    per_animal_results[aid] = {
        'shift': first_shift,
        'baseline': baseline,
        'post': post,
        'trajectory': traj,
        'recovery': recovery,
        'convergence': convergence,
        'comp_early': comp_early,
        'comp_late': comp_late,
        'acc_drop': acc_drop,
        'pse_shift': pse_shift,
        'labels': labels,
    }

    # ── Print summary ───────────────────────────────────────────────────────
    print(f"  Accuracy drop: {acc_drop['value']:.3f} "
          f"(baseline={acc_drop['baseline_mean']:.3f}, "
          f"early post={acc_drop['post_mean']:.3f})")
    print(f"  PSE shift: {pse_shift['value']:.3f}")

    # Recovery speed
    acc_rec = recovery['accuracy']
    if acc_rec['converged']:
        print(f"  Accuracy recovery: tau={acc_rec['tau']:.1f} sessions, "
              f"R²={acc_rec['r_squared']:.2f}")
    else:
        print(f"  Accuracy recovery: fit did not converge")

    ttc = acc_rec.get('sessions_to_criterion', np.nan)
    print(f"  Sessions to criterion (1 SD): {ttc}")

    # Classification
    n_updating = labels.count('updating')
    print(f"  State labels: {n_updating}/{len(labels)} 'updating'")

print(f"\n{'='*60}")
print(f"Analysed {len(per_animal_results)} animals")

## 4. Adaptation Trajectories — Per Animal

In [ ]:
for aid, res in per_animal_results.items():
    traj = res['trajectory']
    n_stats = len(TRACK_STATS)
    n_cols = min(3, n_stats)
    n_rows = int(np.ceil(n_stats / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
    axes_flat = np.array(axes).flatten()

    for ax, stat in zip(axes_flat, TRACK_STATS):
        if stat not in traj.columns:
            ax.set_visible(False)
            continue

        bl = traj[traj['phase'] == 'baseline']
        post = traj[traj['phase'] == 'post']

        ax.plot(bl['relative_session'], bl[stat], 'o-',
                color='grey', alpha=0.6, markersize=4, label='Baseline')
        ax.plot(post['relative_session'], post[stat], 'o-',
                color=COLOURS.get('shift', 'crimson'), alpha=0.8,
                markersize=4, label='Post-shift')

        # Baseline mean line
        bl_mean_col = f'baseline_{stat}_mean'
        if bl_mean_col in traj.columns:
            bl_mean = traj[bl_mean_col].iloc[0]
            if not np.isnan(bl_mean):
                ax.axhline(bl_mean, color='grey', ls='--', alpha=0.4)

        # Recovery fit
        rec = res['recovery'].get(stat, {})
        if rec.get('converged', False) and len(rec.get('fitted_values', [])) > 0:
            ax.plot(rec['relative_sessions'], rec['fitted_values'],
                    '--', color='navy', alpha=0.5, lw=1.5,
                    label=f'τ={rec["tau"]:.1f}')

        ax.axvline(0, color='red', ls=':', alpha=0.4)
        ax.set_xlabel('Relative session')
        ax.set_ylabel(stat)
        ax.set_title(stat)
        ax.legend(fontsize=7)

    for j in range(n_stats, len(axes_flat)):
        axes_flat[j].set_visible(False)

    shift_desc = res['shift']['details']
    fig.suptitle(f"{aid}: {shift_desc.get('before', '?')} → {shift_desc.get('after', '?')}",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Phase Comparisons

In [ ]:
for aid, res in per_animal_results.items():
    print(f"\n{aid}: {res['shift']['details']}")

    print("\n  Baseline vs Early Post:")
    df = res['comp_early']
    for _, row in df.iterrows():
        sig = '*' if row['p_value'] < 0.05 else ' '
        print(f"    {row['stat']:>20s}: {row['a_mean']:.3f} → {row['b_mean']:.3f}  "
              f"p={row['p_value']:.3g} {sig}  d={row['cliffs_delta']:.2f}")

    print("\n  Baseline vs Late Post:")
    df = res['comp_late']
    for _, row in df.iterrows():
        sig = '*' if row['p_value'] < 0.05 else ' '
        print(f"    {row['stat']:>20s}: {row['a_mean']:.3f} → {row['b_mean']:.3f}  "
              f"p={row['p_value']:.3g} {sig}  d={row['cliffs_delta']:.2f}")

## 6. Convergence Summary Table

In [ ]:
# Build summary across animals
summary_rows = []

for aid, res in per_animal_results.items():
    row = {
        'animal_id': aid,
        'shift_type': res['shift']['details'].get('after', '?'),
        'n_baseline': len(res['baseline']),
        'n_post': len(res['post']),
        'accuracy_drop': res['acc_drop']['value'],
        'pse_shift': res['pse_shift']['value'],
    }

    conv = res['convergence']
    for _, crow in conv.iterrows():
        stat = crow['stat']
        row[f'{stat}_tau'] = crow['tau']
        row[f'{stat}_ttc'] = crow['sessions_to_criterion']
        row[f'{stat}_r2'] = crow['r_squared']

    n_updating = res['labels'].count('updating')
    row['n_updating_sessions'] = n_updating
    row['frac_updating'] = n_updating / len(res['labels']) if res['labels'] else np.nan

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("Convergence Summary:")
display_cols = ['animal_id', 'shift_type', 'n_baseline', 'n_post',
                'accuracy_drop', 'accuracy_tau', 'accuracy_ttc',
                'pse_shift', 'n_updating_sessions', 'frac_updating']
display_cols = [c for c in display_cols if c in summary_df.columns]
print(summary_df[display_cols].to_string(index=False, float_format='{:.3f}'.format))

## 7. Group-Level Trajectories

Aggregate shift-aligned trajectories across animals.

In [ ]:
# Collect all trajectories with animal_id
all_trajectories = []
for aid, res in per_animal_results.items():
    traj = res['trajectory'].copy()
    traj['animal_id'] = aid
    all_trajectories.append(traj)

if all_trajectories:
    agg = aggregate_trajectories(all_trajectories, stats=TRACK_STATS,
                                  session_range=(-10, 20))

    n_stats = len(TRACK_STATS)
    n_cols = min(3, n_stats)
    n_rows = int(np.ceil(n_stats / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
    axes_flat = np.array(axes).flatten()

    for ax, stat in zip(axes_flat, TRACK_STATS):
        sub = agg[agg['stat'] == stat]
        if len(sub) == 0:
            ax.set_visible(False)
            continue

        pre = sub[sub['relative_session'] < 0]
        post = sub[sub['relative_session'] >= 0]

        ax.fill_between(pre['relative_session'],
                         pre['mean'] - pre['sem'],
                         pre['mean'] + pre['sem'],
                         color='grey', alpha=0.2)
        ax.plot(pre['relative_session'], pre['mean'],
                'o-', color='grey', markersize=3)

        ax.fill_between(post['relative_session'],
                         post['mean'] - post['sem'],
                         post['mean'] + post['sem'],
                         color=COLOURS.get('shift', 'crimson'), alpha=0.2)
        ax.plot(post['relative_session'], post['mean'],
                'o-', color=COLOURS.get('shift', 'crimson'), markersize=3)

        ax.axvline(0, color='red', ls=':', alpha=0.4)
        ax.set_xlabel('Relative session')
        ax.set_ylabel(stat)
        ax.set_title(f'{stat} (n={int(sub["n_animals"].max())} animals)')

    for j in range(n_stats, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle('Group-Level Adaptation Trajectories (mean ± SEM)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No animals with shift data to aggregate.')

## 8. Update Matrix Evolution Across the Shift

How does the update matrix change from expert baseline through
early post-shift to late post-shift?

In [ ]:
for aid, res in per_animal_results.items():
    baseline = res['baseline']
    post = res['post']

    phases = [
        ('Baseline (last 5)', baseline[-5:]),
        ('Early post (1–3)', post[:3]),
        ('Mid post', post[3:8] if len(post) > 8 else post[3:]),
        ('Late post (last 5)', post[-5:]),
    ]
    phases = [(name, sess) for name, sess in phases if len(sess) > 0]

    fig, axes = plt.subplots(1, len(phases), figsize=(4 * len(phases), 4))
    if len(phases) == 1:
        axes = [axes]

    ums = []
    for name, sessions in phases:
        all_stim, all_ch, all_cat = [], [], []
        for s in sessions:
            arrays = s.trials.get_arrays(exclude_abort=True, exclude_opto=True)
            v = ~arrays['no_response']
            all_stim.append(arrays['stimuli'][v])
            all_ch.append(arrays['choices'][v])
            all_cat.append(arrays['categories'][v])
        stim = np.concatenate(all_stim)
        ch = np.concatenate(all_ch)
        cat = np.concatenate(all_cat)
        um, _, _ = compute_update_matrix(stim, ch, cat, n_bins=8)
        ums.append(um)

    vmax = max(np.nanmax(np.abs(um)) for um in ums if not np.all(np.isnan(um)))

    for ax, (name, _), um in zip(axes, phases, ums):
        plot_update_matrix(um, ax=ax, vmin=-vmax, vmax=vmax)
        ax.set_title(name, fontsize=9)

    shift_desc = res['shift']['details']
    fig.suptitle(f"{aid}: UM across shift "
                 f"({shift_desc.get('before','?')} → {shift_desc.get('after','?')})",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Save Results

In [ ]:
import pickle

results_dir = Path('../results/adaptation')
results_dir.mkdir(parents=True, exist_ok=True)

# Save summary table
summary_df.to_csv(results_dir / 'adaptation_summary.csv', index=False)

# Save per-animal results (excluding session objects)
save_results = {}
for aid, res in per_animal_results.items():
    save_results[aid] = {
        'shift': res['shift'],
        'trajectory': res['trajectory'],
        'convergence': res['convergence'],
        'comp_early': res['comp_early'],
        'comp_late': res['comp_late'],
        'acc_drop': res['acc_drop'],
        'pse_shift': res['pse_shift'],
        'labels': res['labels'],
    }

with open(results_dir / 'adaptation_results.pkl', 'wb') as f:
    pickle.dump(save_results, f)

print(f"Saved to {results_dir}/")

## Interpretation

**Accuracy drop**: Immediate effect of the distribution shift. Larger drops
indicate the old statistical model was more inadequate for the new distribution.

**Recovery τ**: How many sessions to re-adapt. Smaller τ = faster adaptation.
Compare across transition types: Uniform→Hard-A (novel) should be slower
than Hard-A→Hard-B (with meta-experience).

**State classification**: Sessions classified as 'updating' are candidates
for when PPC inactivation should have maximum effect (Aim 2 prediction).

**Update matrix evolution**: Shows whether serial dependence structure changes
during adaptation — critical for linking to BE/SC model predictions.

**Next steps**:
- 4b: Fit BE/SC models to post-shift data to track parameter trajectories
- 5: Use SLDS on these features for automated learning phase assignment
- 6a: Generate opto predictions from the adaptation dynamics